In [1]:
import torch
import torch.nn as nn
import numpy as np
import polars as pl
import glob
import os

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score


In [2]:
# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
# LOAD CHUNKS (MANUAL FIXED SPLIT)
def get_chunk_number(path):
    return int(path.split("_")[-1].replace(".parquet", ""))

all_chunks = sorted(
    glob.glob("DataSet/6_class_chunks/*.parquet"),
    key=get_chunk_number
)

print("Total chunks found:", len(all_chunks))

# ---- SAFETY CHECK ----
assert len(all_chunks) >= 164, \
    f"Expected at least 164 chunks, found {len(all_chunks)}"

# ---- FIXED SPLIT ----
train_files = all_chunks[0:148]
test_files  = all_chunks[148:156]
val_files   = all_chunks[156:164]

print("Train:", len(train_files))
print("Test :", len(test_files))
print("Val  :", len(val_files))

Total chunks found: 164
Train: 148
Test : 8
Val  : 8


In [4]:
# LABEL REMAP (1–6 → 0–5)
label_map = {1:0, 2:1, 3:2, 4:3, 5:4, 6:5}

def remap_labels(arr):
    return np.vectorize(label_map.get)(arr)


In [5]:
# CLASS DISTRIBUTION (FOR WEIGHTS)
print("\nCalculating class distribution...")

counts = {}
total = 0

for file in train_files:
    df = pl.read_parquet(file, columns=["Attack"])
    values = df["Attack"].to_numpy()

    unique, freq = np.unique(values, return_counts=True)

    for u, f in zip(unique, freq):
        counts[u] = counts.get(u, 0) + f
        total += f

print("Class counts:", counts)

# Inverse frequency weights
weights = []
for original in sorted(label_map.keys()):
    weights.append(total / counts[original])

weights = torch.tensor(weights, dtype=torch.float32).to(device)

# Normalize (mean = 1)
weights = weights / weights.mean()

weights[5] *= 0.8   

print("Adjusted Class weights:", weights)


Calculating class distribution...
Class counts: {np.int32(1): np.int64(19623107), np.int32(2): np.int64(16129971), np.int32(3): np.int64(129238), np.int32(4): np.int64(104635), np.int32(5): np.int64(3412711), np.int32(6): np.int64(50004)}
Adjusted Class weights: tensor([0.0081, 0.0099, 1.2315, 1.5211, 0.0466, 2.5463], device='cuda:0')


In [6]:
# MODEL
class ResidualMLP(nn.Module):
    def __init__(self, input_dim=37, num_classes=6):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 768)
        self.bn1 = nn.BatchNorm1d(768)
        self.drop1 = nn.Dropout(0.25)

        self.fc2 = nn.Linear(768, 768)
        self.bn2 = nn.BatchNorm1d(768)
        self.drop2 = nn.Dropout(0.25)

        self.fc3 = nn.Linear(768, 512)
        self.bn3 = nn.BatchNorm1d(512)

        self.fc4 = nn.Linear(512, 256)
        self.bn4 = nn.BatchNorm1d(256)

        self.fc5 = nn.Linear(256, 128)
        self.bn5 = nn.BatchNorm1d(128)

        self.out = nn.Linear(128, num_classes)

        self.act = nn.GELU()

    def forward(self, x):

        x1 = self.act(self.bn1(self.fc1(x)))
        x1 = self.drop1(x1)

        x2 = self.act(self.bn2(self.fc2(x1)))
        x2 = x2 + x1
        x2 = self.drop2(x2)

        x3 = self.act(self.bn3(self.fc3(x2)))
        x4 = self.act(self.bn4(self.fc4(x3)))
        x5 = self.act(self.bn5(self.fc5(x4)))

        return self.out(x5)

model = ResidualMLP().to(device)


In [7]:
# LOSS / OPTIMIZER / SCHEDULER
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

EPOCHS = 12

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

scaler = torch.amp.GradScaler("cuda")


In [8]:
# TRAIN FUNCTION
def train_one_epoch(files):

    model.train()
    total_loss = 0
    total_samples = 0

    for file in files:

        df = pl.read_parquet(file)

        X = df.drop("Attack").to_numpy().astype(np.float32)
        y = remap_labels(df["Attack"].to_numpy().astype(np.int64))

        dataset = TensorDataset(
            torch.from_numpy(X),
            torch.from_numpy(y)
        )

        loader = DataLoader(
            dataset,
            batch_size=8192,
            shuffle=True,
            num_workers=0,
            pin_memory=True
        )

        for xb, yb in loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                outputs = model(xb)
                loss = criterion(outputs, yb)

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * xb.size(0)
            total_samples += xb.size(0)

        del df, dataset, loader
        torch.cuda.empty_cache()

    return total_loss / total_samples


In [9]:
# EVALUATION
def evaluate(files):

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for file in files:

            df = pl.read_parquet(file)

            X = df.drop("Attack").to_numpy().astype(np.float32)
            y = remap_labels(df["Attack"].to_numpy().astype(np.int64))

            X_tensor = torch.from_numpy(X).to(device)

            outputs = model(X_tensor)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y)

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return acc, macro_f1

In [10]:
# TRAIN LOOP
os.makedirs("Trained", exist_ok=True)

best_f1 = 0
patience = 3
no_improve = 0

for epoch in range(EPOCHS):

    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    train_loss = train_one_epoch(train_files)
    print(f"Train Loss: {train_loss:.6f}")

    val_acc, val_f1 = evaluate(val_files)
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Validation Macro F1: {val_f1:.4f}")

    scheduler.step()

    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve = 0

        save_path = f"Trained/IDS-MLP-6class-epoch{epoch+1}-F1_{val_f1:.4f}.pth"
        torch.save(model.state_dict(), save_path)

        print("✅ Best model saved")

    else:
        no_improve += 1

    if no_improve >= patience:
        print("Early stopping triggered")
        break

print("\nTraining Completed")
print("Best Macro F1:", best_f1)


===== Epoch 1/12 =====
Train Loss: 0.032804
Validation Accuracy: 0.9932
Validation Macro F1: 0.9802
✅ Best model saved

===== Epoch 2/12 =====
Train Loss: 0.023828
Validation Accuracy: 0.9936
Validation Macro F1: 0.9804
✅ Best model saved

===== Epoch 3/12 =====
Train Loss: 0.022587
Validation Accuracy: 0.9938
Validation Macro F1: 0.9831
✅ Best model saved

===== Epoch 4/12 =====
Train Loss: 0.021884
Validation Accuracy: 0.9939
Validation Macro F1: 0.9852
✅ Best model saved

===== Epoch 5/12 =====
Train Loss: 0.021307
Validation Accuracy: 0.9941
Validation Macro F1: 0.9856
✅ Best model saved

===== Epoch 6/12 =====
Train Loss: 0.020824
Validation Accuracy: 0.9940
Validation Macro F1: 0.9862
✅ Best model saved

===== Epoch 7/12 =====
Train Loss: 0.020508
Validation Accuracy: 0.9942
Validation Macro F1: 0.9862
✅ Best model saved

===== Epoch 8/12 =====
Train Loss: 0.020253
Validation Accuracy: 0.9942
Validation Macro F1: 0.9864
✅ Best model saved

===== Epoch 9/12 =====
Train Loss: 0.01

In [11]:
import glob

# ---- FIND LATEST SAVED MODEL ----
saved_models = sorted(glob.glob("Trained/IDS-MLP-6class-epoch*.pth"))

assert len(saved_models) > 0, "No saved model found in Trained/"

last_model_path = saved_models[-1]
print("Loading:", last_model_path)

model.load_state_dict(torch.load(last_model_path))
model.to(device)

# OPTIONAL: reset optimizer LR if needed
for param_group in optimizer.param_groups:
    param_group['lr'] = 3e-4   # slightly lower LR for fine-tuning

# Continue for extra epochs
EXTRA_EPOCHS = 8

best_f1 = 0
patience = 3
no_improve = 0

for epoch in range(EXTRA_EPOCHS):

    print(f"\n===== Continue Epoch {epoch+1}/{EXTRA_EPOCHS} =====")

    train_loss = train_one_epoch(train_files)
    print(f"Train Loss: {train_loss:.6f}")

    val_acc, val_f1 = evaluate(val_files)
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Validation Macro F1: {val_f1:.4f}")

    scheduler.step()

    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve = 0

        save_path = f"Trained/IDS-MLP-6class-continued-epoch{epoch+1}-F1_{val_f1:.4f}.pth"
        torch.save(model.state_dict(), save_path)

        print("✅ New Best model saved")

    else:
        no_improve += 1

    if no_improve >= patience:
        print("Early stopping triggered")
        break

print("\nContinue Training Completed")
print("Best Continued Macro F1:", best_f1)

Loading: Trained/IDS-MLP-6class-epoch9-F1_0.9866.pth

===== Continue Epoch 1/8 =====


/tmp/ipykernel_21827/2626611044.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(last_model_path))


Train Loss: 0.020463
Validation Accuracy: 0.9942
Validation Macro F1: 0.9864
✅ New Best model saved

===== Continue Epoch 2/8 =====
Train Loss: 0.020274
Validation Accuracy: 0.9942
Validation Macro F1: 0.9864
✅ New Best model saved

===== Continue Epoch 3/8 =====
Train Loss: 0.024319
Validation Accuracy: 0.9938
Validation Macro F1: 0.9831

===== Continue Epoch 4/8 =====
Train Loss: 0.025219
Validation Accuracy: 0.9937
Validation Macro F1: 0.9822

===== Continue Epoch 5/8 =====
Train Loss: 0.026625
Validation Accuracy: 0.9936
Validation Macro F1: 0.9803
Early stopping triggered

Continue Training Completed
Best Continued Macro F1: 0.986440985986055


In [ ]:
import torch
import polars as pl
import glob
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# =====================================================
# CONFIG
# =====================================================
DATA_PATH = "DataSet/6_class_chunks/*.parquet"
MODEL_PATH = "models/6_class/IDS-MLP-6class-epoch10-F1_0.9874.pth"
BATCH_SIZE = 4096

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =====================================================
# SELECT LAST 8 FILES FOR TEST
# =====================================================
all_files = sorted(glob.glob(DATA_PATH))
test_files = all_files[-8:]

print("Total files:", len(all_files))
print("Using for test:", len(test_files))

# =====================================================
# LOAD TEST DATA
# =====================================================
print("\nLoading test data...")

test_df = pl.scan_parquet(test_files).collect()

# IMPORTANT: Remove class 0 if it exists
test_df = test_df.filter(pl.col("Attack") != 0)

# Re-encode labels to match training (shift by -1)
y_test = test_df["Attack"].to_numpy() - 1

# Features
X_test = test_df.drop("Attack").to_numpy()

# Convert to torch
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

test_dataset = TensorDataset(X_test, y_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Test samples:", len(test_dataset))

# =====================================================
# LOAD MODEL
# =====================================================
model.load_state_dict(
    torch.load(MODEL_PATH, weights_only=True)
)

model.to(device)
model.eval()

# =====================================================
# TESTING
# =====================================================
print("\nRunning evaluation...")

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# =====================================================
# METRICS
# =====================================================
accuracy = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average="macro")

print("\n===== TEST RESULTS =====")
print("Test Accuracy:", accuracy)
print("Test Macro F1:", macro_f1)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Total samples per true class (row sum)
row_sums = cm.sum(axis=1)

# Total predictions per predicted class (column sum)
col_sums = cm.sum(axis=0)

class_id = 5

print("===================================")
print("1️⃣ When TRUE class = 5")
print("===================================")

for pred_class in range(len(cm)):
    percentage = (cm[class_id, pred_class] / row_sums[class_id]) * 100
    print(f"Predicted as {pred_class}: {percentage:.2f}%")

print("\n===================================")
print("2️⃣ When PREDICTED class = 5")
print("===================================")

for true_class in range(len(cm)):
    percentage = (cm[true_class, class_id] / col_sums[class_id]) * 100
    print(f"Actually class {true_class}: {percentage:.2f}%")

In [24]:
import polars as pl

lf = pl.scan_parquet("DataSet/6_class_chunks/*.parquet")

class_freq = (
    lf.group_by("Attack")
      .agg(pl.len().alias("count"))
      .sort("Attack")
)

print(class_freq.collect())

shape: (6, 2)
┌────────┬──────────┐
│ Attack ┆ count    │
│ ---    ┆ ---      │
│ i32    ┆ u32      │
╞════════╪══════════╡
│ 1      ┆ 21743661 │
│ 2      ┆ 17875560 │
│ 3      ┆ 143097   │
│ 4      ┆ 115969   │
│ 5      ┆ 3781417  │
│ 6      ┆ 55431    │
└────────┴──────────┘
